# All library you need

In [6]:
import pandas as pd
import numpy as np
import json
import pickle
import streamlit as st
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from textblob import TextBlob
from tqdm import tqdm

# ----------------------
# Step 1: Load & Process Yelp Data (pre-exported CSVs)
# ----------------------

In [7]:

# --- Load only necessary columns from business.json ---
def load_business_subset(file_path):
    df = pd.read_json(file_path, lines=True, dtype={"categories": str})
    df = df[df['categories'].str.contains("Restaurants", na=False)]
    df = df[[
        'business_id', 'city', 'categories', 'stars', 'attributes'
    ]]
    return df

# --- Clean attributes ---
def clean_attributes(df):
    def extract(attr):
        try:
            return json.loads(attr.replace("u'", "'").replace("'", '"'))
        except:
            return {}

    # Convert 'attributes' column to a DataFrame
    attr_df = df['attributes'].dropna().apply(extract).apply(pd.Series)

    # Extract 'has_parking' safely
    def extract_parking(val):
        try:
            parsed = json.loads(val.replace("u'", "'").replace("'", '"'))
            return any(parsed.values())
        except:
            return False

    attr_df["has_parking"] = df['attributes'].dropna().apply(lambda x: extract_parking(x))

    # Safely get the desired columns if they exist
    keep_cols = ["RestaurantsDelivery", "RestaurantsTakeOut", "BusinessAcceptsCreditCards",
                 "Caters", "OutdoorSeating", "RestaurantsPriceRange2", "WiFi", "Alcohol", "has_parking"]

    for col in keep_cols:
        if col not in attr_df.columns:
            attr_df[col] = None

    # Merge back with business dataframe
    df_cleaned = pd.concat([df.drop(columns='attributes'), attr_df[keep_cols]], axis=1)

    # Convert booleans
    bool_cols = ["RestaurantsDelivery", "RestaurantsTakeOut", "BusinessAcceptsCreditCards",
                 "Caters", "OutdoorSeating", "has_parking"]
    for col in bool_cols:
        df_cleaned[col] = df_cleaned[col].map({'True': True, 'False': False}).fillna(False)

    df_cleaned['RestaurantsPriceRange2'] = df_cleaned['RestaurantsPriceRange2'].fillna(2).astype(int)

    return df_cleaned

# --- Load checkins ---
def load_checkins(file_path):
    df = pd.read_json(file_path, lines=True)
    return df['date'].str.split(',').explode().groupby(df['business_id']).count().reset_index(name='checkins_per_year')

# --- Process reviews in chunks ---
def process_reviews(file_path, target_business_ids):
    sentiment_sum = {}
    review_count = {}

    # First, count total lines for tqdm
    total_lines = sum(1 for _ in open(file_path, 'r'))

    with open(file_path, 'r') as f:
        for line in tqdm(f, total=total_lines, desc="Processing Reviews"):
            data = json.loads(line)
            business_id = data['business_id']

            if business_id not in target_business_ids:
                continue

            text = str(data.get('text', ''))
            sentiment = TextBlob(text).sentiment.polarity

            sentiment_sum[business_id] = sentiment_sum.get(business_id, 0) + sentiment
            review_count[business_id] = review_count.get(business_id, 0) + 1

    # Convert to DataFrames
    sentiment_avg = {k: sentiment_sum[k] / review_count[k] for k in sentiment_sum}
    sentiment_df = pd.DataFrame(list(sentiment_avg.items()), columns=['business_id', 'sentiment'])
    review_df = pd.DataFrame(list(review_count.items()), columns=['business_id', 'reviews_per_year'])
    return sentiment_df, review_df


# ---------------------
# Run optimized pipeline
# ---------------------
print("Loading business.json...")
business_df = load_business_subset("business.json")
print("Cleaning attributes...")
business_df = clean_attributes(business_df)

print("Loading checkin.json...")
checkin_df = load_checkins("checkin.json")

print("Processing review.json...")
business_ids = set(business_df['business_id'])
sentiment_df, review_count_df = process_reviews("review.json", business_ids)


# Merge all
df = business_df.merge(checkin_df, on='business_id', how='left')
df = df.merge(sentiment_df, on='business_id', how='left')
df = df.merge(review_count_df, on='business_id', how='left')

df = df.fillna({"checkins_per_year": 0, "sentiment": 0, "reviews_per_year": 0})

# Rename and simplify
df = df.rename(columns={
    'RestaurantsDelivery': 'has_delivery',
    'RestaurantsTakeOut': 'has_takeout',
    'BusinessAcceptsCreditCards': 'has_creditcard',
    'Caters': 'caters',
    'OutdoorSeating': 'outdoor_seating',
    'RestaurantsPriceRange2': 'price_range',
    'WiFi': 'has_wifi',
    'Alcohol': 'alcohol',
    'categories': 'category'
})
df['category'] = df['category'].str.split(',').str[0]  # simplify

print(f"✅ Final shape: {df.shape}")
df['category'] = df['category'].str.split(',').str[0]  # simplify

Loading business.json...
Cleaning attributes...


/var/folders/5j/69rtr8094_j8kqd_my2zxqr00000gn/T/ipykernel_23674/3983004016.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_cleaned[col] = df_cleaned[col].map({'True': True, 'False': False}).fillna(False)
/var/folders/5j/69rtr8094_j8kqd_my2zxqr00000gn/T/ipykernel_23674/3983004016.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_cleaned[col] = df_cleaned[col].map({'True': True, 'False': False}).fillna(False)
/var/folders/5j/69rtr8094_j8kqd_my2zxqr00000gn/T/ipykernel_23674/3983004016.py:50: FutureWarning: Downcasting object dtype arrays on .

Loading checkin.json...
Processing review.json...


Processing Reviews: 100%|███████████| 6990280/6990280 [50:08<00:00, 2323.61it/s]


✅ Final shape: (52268, 16)


# ----------------------
# Step 2: Train Model
# ----------------------

In [8]:
df.head(10)

,business_id,city,category,stars,has_delivery,has_takeout,has_creditcard,caters,outdoor_seating,price_range,has_wifi,alcohol,has_parking,checkins_per_year,sentiment,reviews_per_year
0,MTSW4McQd7CbVtyjqoe9mw,Philadelphia,Restaurants,4.0,False,False,False,False,False,2,None,None,False,335.0,0.280990,87
1,CF33F8-E6oudUQ46HnavjQ,Ashland City,Burgers,2.0,False,False,False,False,False,2,None,None,False,22.0,-0.136909,6
2,k0hlBqXX-Bt0vf1op7Jr1w,Affton,Pubs,3.0,False,False,False,False,False,2,None,None,False,40.0,0.187153,19
3,bBDDEgkFA1Otx9Lfe7BZUQ,Nashville,Ice Cream & Frozen Yogurt,1.5,False,False,False,False,False,2,None,None,False,21.0,0.058651,10
4,eEOYSgkmpB90uNA7lDOMRA,Tampa Bay,Vietnamese,4.0,False,False,False,False,False,2,None,None,False,4.0,0.196803,11
5,il_Ro8jwPlHresjw9EGmBg,Indianapolis,American (Traditional),2.5,False,False,False,False,False,2,None,None,False,54.0,0.052765,29
6,0bPLkL0QhhPO5kt1_EXmNQ,Largo,Food,4.5,False,False,False,False,False,2,None,None,False,264.0,0.294617,106
7,MUTTqe8uqyMdBl186RmNeA,Philadelphia,Sushi Bars,4.0,False,False,False,False,False,2,None,None,False,172.0,0.343521,250
8,ROeacJQwBeh05Rqg7F6TCg,Philadelphia,Korean,4.5,False,False,False,False,False,2,None,None,False,221.0,0.298164,208
9,WKMJwqnfZKsAae75RMP6jA,Edmonton,Coffee & Tea,4.0,False,False,False,False,False,2,None,None,False,226.0,0.243915,40


In [9]:
features = [
    'city', 'category', 'reviews_per_year', 'checkins_per_year', 'sentiment',
    'has_delivery', 'has_takeout', 'has_creditcard', 'caters', 'outdoor_seating',
    'price_range', 'has_parking', 'has_wifi', 'alcohol']

X = df[features]
y = df['stars']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['city', 'category', 'has_wifi', 'alcohol'])
], remainder='passthrough')

model = Pipeline([
    ('prep', preprocessor),
    ('reg', RandomForestRegressor(n_estimators=100, random_state=42))
])

model.fit(X, y)

# Save model
with open("stars_predictor.pkl", "wb") as f:
    pickle.dump(model, f)



# ----------------------
# Step 3: Back-end of Streamlit App
# ----------------------

In [11]:
# Preparing app inputs

# Load business data
business_df = pd.read_json("business.json", lines=True)

# Filter for Restaurants only
restaurant_df = business_df[business_df['categories'].str.contains("Restaurants", na=False)]

# Keep only business_id, city, and category
restaurant_df = restaurant_df[['business_id', 'city', 'categories']]
restaurant_df = restaurant_df.rename(columns={'categories': 'category'})

# Save to parquet so App can use these data for inputs
restaurant_df[['city', 'category']].dropna().drop_duplicates().to_parquet("processed_restaurant_data.parquet")

print("✅ processed_restaurant_data.parquet created.")


✅ processed_restaurant_data.parquet created.
